In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
import random

import numpy as np
import torch

from torch.utils.data import IterableDataset
from torch.utils.data import DataLoader

from configs.config import config

In [3]:
token_file = config.train_file.parent / "tokens.bin"

print(token_file)
print(token_file.exists())

C:\Users\FH\Documents\PersionLLM\PersianGPT\data\processed\tokens.bin
True


In [4]:
tokens = np.memmap(
    token_file,
    dtype=np.uint16,
    mode="r"
)

print(tokens)
print()

print("Total Tokens :", len(tokens))

[ 6083  1738    80 ...   186 31867     2]

Total Tokens : 265009258


In [5]:
print(tokens[:20])
print(tokens[100:120])
print(tokens[-20:])

[ 6083  1738    80 31879 15127 18341 11433  9144  6083  1738    80  3526
  2118  6307  6298  5850  4572  8947  3438  2776]
[16756  5258   226   418    29  1817   545  5804   105  9070    63    29
 31857   487  2033   226    27 13885   842   226]
[  937    35  5820    43    48  1682  8198    35  1109    22   646  2197
     9   874   676   105  2770   186 31867     2]


In [6]:
class PersianDataset(IterableDataset):

    def __init__(self, tokens, seq_len):

        self.tokens = tokens
        self.seq_len = seq_len

    def __iter__(self):

        max_start = len(self.tokens) - self.seq_len - 1

        while True:

            start = random.randint(0, max_start)

            x = self.tokens[start:start+self.seq_len]

            y = self.tokens[start+1:start+self.seq_len+1]

            yield (
                torch.tensor(x, dtype=torch.long),
                torch.tensor(y, dtype=torch.long)
            )

In [7]:
dataset = PersianDataset(
    tokens=tokens,
    seq_len=config.max_seq_len
)

print(dataset)

In [8]:
dataloader = DataLoader(
    dataset,
    batch_size=config.batch_size,
    num_workers=0,
)

In [9]:
batch = next(iter(dataloader))

input_ids, target_ids = batch

print(input_ids.shape)
print(target_ids.shape)

torch.Size([16, 256])
torch.Size([16, 256])


In [10]:
print("Input")
print(input_ids[0])

print()

print("Target")
print(target_ids[0])

Input
tensor([    2,   196,   679,    72,   646,  1037,     9,  4343,  8272,  7775,
        31889,  3246,  4683,   465,  7200, 31931, 11237,   970,  7200,  7775,
         3290,   287,   148,  6403,     2,   196,   972,   300,  3464,    29,
         2217, 30269,    27,  2415,  4880,   582,  4952,   892,  5045,   118,
        25548,    18,  1220, 31889,   523,  2883, 25630,  8272,     2,   196,
          679,  4854,     9,  6569,   743,  1077,  2217, 30269,   582,   646,
         6569,   365, 27037,  2013, 31889,    82,   646,  1037,  7200,     2,
          196,   972,   300,   646,  1116,   365, 14885,  5693,    18,   872,
         2947, 31931,  1156,  1116, 14885,  2013,    38,   646,  2975,     2,
          196,   972, 13964,  9023,  1123,  6755,  5693, 31889,    22,   679,
          872,  2947,  2738,     2,   196,   972,   930,   646,  8272,   582,
         7511,  2947,    82,  2287,    29,  2217, 30269,  2542,   582,  3364,
          679,  1077,  2217, 30269,     2,   196,   622,  

In [11]:
print(torch.equal(
    input_ids[0][1:],
    target_ids[0][:-1]
))

True


In [12]:
print(input_ids.dtype)
print(target_ids.dtype)

print(input_ids.device)

torch.int64
torch.int64
cpu
